## This notebook is used to analyze the data under all 9 settings

In [ ]:
import pandas as pd


dataset = pd.read_csv('../processed_data/NGQA_benchmark.csv')

In [ ]:
import ast
import networkx as nx


def convert_to_networkx(row):
    node_list = ast.literal_eval(row['node_list'])
    edge_list = ast.literal_eval(row['edge_list'])
    
    # Create NetworkX graph
    graph = nx.Graph()

    # Add nodes with attributes
    for node in node_list:
        graph.add_node(node[0], **node[1])

    # Add edges with attributes
    for edge in edge_list:
        if len(edge) == 3:  # [source, relationship, target]
            graph.add_edge(edge[0], edge[2], relationship=edge[1])
    return graph


def average_snr(data):
    '''
    This function calculates the average (number of useful nodes / number of total nodes) among all the graphs in the dataset.
    Also, it will count the average number of the edges.
    '''
    snr = 0
    useful_node_count = 0
    node_count = 0
    edge_count = 0
    for _, row in data.iterrows():
        g = convert_to_networkx(row)
        optimal_nodes = set()
        optimal_paths = nx.all_simple_paths(g, 0, 1)
        for path in optimal_paths:
            for i in path:
                optimal_nodes.add(i)
        snr += len(optimal_nodes) / len(g.nodes)
        useful_node_count += len(optimal_nodes)
        node_count += len(g.nodes)
        edge_count += len(g.edges)
        
    avg_snr = snr / len(data)
    avg_useful_nodes = useful_node_count / len(data)
    avg_total_nodes = node_count / len(data)
    avg_edge_count = edge_count / len(data)
    return avg_useful_nodes, avg_total_nodes, avg_edge_count, avg_snr


def average_tag_snr(data):
    '''
    This function calculates the average (number of useful tags / number of total tags) among all the graphs in the dataset.
    '''
    tag_snr = 0
    tag_count = 0
    usefl_tag_count = 0
    for _, row in data.iterrows():
        
        # Count the useful_tags
        g = convert_to_networkx(row)
        paths = nx.all_simple_paths(g, 0, 1)
        useful_tags = set()
        for path in paths:
            for i in path:
                if g.nodes[i]['name'] == 'food_nutrition_tag':
                    useful_tags.add(i)
        usefl_tag_count += len(useful_tags)
              
        # Count the total number of nutrition tags
        total_tags = [i for i in g.nodes if g.nodes[i]['name'] == 'food_nutrition_tag']
        tag_count += len(total_tags)
        
        tag_snr += len(useful_tags) / len(total_tags)
    
    avg_tag_count = tag_count / len(data)
    avg_useful_tag_count = usefl_tag_count / len(data)
    avg_tag_snr = tag_snr / len(data)
    return avg_useful_tag_count, avg_tag_count, avg_tag_snr



In [ ]:
levels = ['easy', 'medium', 'hard']

for level in levels:
    data = dataset[dataset['difficulty'] == level]
    avg_useful_nodes, avg_total_nodes, avg_edge_count, avg_snr = average_snr(data)
    avg_useful_tag_count, avg_tag_count, avg_tag_snr = average_tag_snr(data)
    print(f'Level: {level}')
    print(f"Class ratio: 'Yes' {data['answer_easy'].value_counts()['Yes'] / len(data)}, 'No' {data['answer_easy'].value_counts()['No'] / len(data)}")
    print(f'Average useful nodes: {avg_useful_nodes}')
    print(f'Average total nodes: {avg_total_nodes}')
    print(f'Average edge count: {avg_edge_count}')
    print(f'Average node SNR: {avg_snr}')
    print(f'Average useful tags: {avg_useful_tag_count}')
    print(f'Average total tags: {avg_tag_count}')
    print(f'Average tag SNR: {avg_tag_snr}')
    
    print()

## Count the unique users

In [ ]:
import ast

user_set = set()

for _, row in dataset.iterrows():
    for [node_id, node_data] in ast.literal_eval(row['node_list']):
        if node_id == 0:
            user_set.add(node_data['name'])
            continue
        
print(len(user_set))